In [50]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

In [51]:
import os
os.path.abspath("")


'/Users/jmoses2013/Desktop/Commuting by County Data'

In [52]:
# Import csv files and combine
df06_10 = pd.read_csv("/Users/jmoses2013/Desktop/Commuting by County Data/table1-2006-2010.csv")
df11_15 = pd.read_csv("/Users/jmoses2013/Desktop/Commuting by County Data/table1-2011-2015.csv")

df06_10['Period'] = '2006-2010'
df11_15['Period'] = '2011-2015'

df = pd.concat([df06_10, df11_15], ignore_index=True)
df.head()

,Residence State FIPS Code,Residence County FIPS Code,Residence State,Residence County,Workplace State FIPS Code,Workplace County FIPS Code,Workplace State,Workplace County,Workers in Commuting Flow,MOE,Period,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,1,1,Alabama,Autauga County,1,1,Alabama,Autauga County,"8,768",602,2006-2010,NaN,NaN,NaN,NaN
1,1,1,Alabama,Autauga County,1,7,Alabama,Bibb County,15,21,2006-2010,NaN,NaN,NaN,NaN
2,1,1,Alabama,Autauga County,1,13,Alabama,Butler County,15,20,2006-2010,NaN,NaN,NaN,NaN
3,1,1,Alabama,Autauga County,1,15,Alabama,Calhoun County,10,15,2006-2010,NaN,NaN,NaN,NaN
4,1,1,Alabama,Autauga County,1,21,Alabama,Chilton County,385,105,2006-2010,NaN,NaN,NaN,NaN


In [53]:
# Clean Data
df.loc[df['Workplace State'] == 'Canada','Workplace FIPS Code'] = 'CC'
df.loc[df['Workplace State'] == 'Mexico','Workplace FIPS Code'] = 'MX'



In [54]:
# Pivot out state values
state_pivot = df.pivot_table(index=['Residence State', 'Workplace State'],
                       columns='Period',
                       values='Workers in Commuting Flow',
                            aggfunc = sum)                              
state_pivot['2011-2015'] = pd.to_numeric(state_pivot['2011-2015'], errors='coerce')
state_pivot['2006-2010'] = pd.to_numeric(state_pivot['2006-2010'], errors='coerce')
state_pivot['change_abs'] = state_pivot['2011-2015'] - state_pivot['2006-2010']
state_pivot['change_pct'] = (state_pivot['change_abs'] / state_pivot['2006-2010']) * 100
state_pivot.head()


Period                              2006-2010     2011-2015    change_abs  \
Residence State Workplace State                                             
Alabama         Afghanistan      1.063826e+13           NaN           NaN   
                Africa           1.235121e+07           NaN           NaN   
                Alabama                   NaN           NaN           NaN   
                Alaska           1.486142e+12  1.215332e+09 -1.484927e+12   
                Angola           6.000000e+00           NaN           NaN   

Period                           change_pct  
Residence State Workplace State              
Alabama         Afghanistan             NaN  
                Africa                  NaN  
                Alabama                 NaN  
                Alaska           -99.918222  
                Angola                  NaN

In [71]:
county_pivot = df.pivot_table(index=['Residence State', 'Residence County', 'Workplace State', 'Workplace County'],
                              columns='Period',
                       values='Workers in Commuting Flow',
                            aggfunc = sum)   

In [62]:
# Net outbound change by State
state_pivot.groupby('Residence State')['change_abs'].sum()

Residence State
Alabama                 9.139884e+190
Alaska                  -1.472517e+19
Arizona                 1.611623e+105
Arkansas                3.199186e+236
California              1.470617e+292
Colorado                3.011231e+212
Connecticut             5.200220e+110
Delaware                 3.725264e+51
District of Columbia    -1.740521e+17
Florida                -1.191972e+299
Georgia                 4.625417e+224
Hawaii                  -5.537838e+56
Idaho                   2.026331e+123
Illinois                4.558472e+155
Indiana                 -3.567641e+95
Iowa                   -2.811955e+292
Kansas                  2.242150e+236
Kentucky                1.120289e+167
Louisiana              -3.043381e+152
Maine                   -9.672343e+75
Maryland               -1.526827e+156
Massachusetts          -1.712119e+303
Michigan               -4.443710e+176
Minnesota              -6.611257e+110
Mississippi             7.148932e+304
Missouri                8.229895e+

In [63]:
df_1= df[df['Period'] == '2006-2010']
df_2 = df[df['Period'] == '2011-2015']


In [64]:
G_1 = nx.DiGraph()
G_2 = nx.DiGraph()

In [65]:
for _, row in df_1.iterrows():
    G_1.add_edge(row['Residence County FIPS Code'],
                 row['Workplace County FIPS Code'],
                 weight=row['Workers in Commuting Flow'])
for _, row in df_2.iterrows():
    G_2.add_edge(row['Residence County FIPS Code'],
                 row['Workplace County FIPS Code'],
                 weight=row['Workers in Commuting Flow'])

In [ ]:
# Filter for California FIP Codes
subset_fips = [fips for fips in G_pre.nodes if str(fips).startswith('06')]
subgraph_pre = G_pre.subgraph(subset_fips)
subgraph_post = G_post.subgraph(subset_fips)
